<a href="https://colab.research.google.com/github/Jhanmoreno1/Actividad3/blob/main/Copia_de_Moreno_Jhan_Actividad_3_Big_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# INSTALACIÓN LIMPIA DE SPARK 3.5.0
# ============================================

print("1. Instalando Java 11...")
!apt-get install -qq openjdk-11-jdk-headless

print("2. Descargando Spark 3.5.0...")
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz

print("3. Descomprimiendo Spark...")
!tar -xzf spark-3.5.0-bin-hadoop3.tgz

print("4. Instalando paquetes Python...")
!pip install -q pyspark==3.5.0 findspark

print("5. Configurando variables de entorno...")
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

print("6. Inicializando Spark...")
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Actividad_2") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .master("local[*]") \
    .getOrCreate()

print("\n" + "="*50)
print("✅ CONFIGURACIÓN EXITOSA")
print("="*50)
print(f"Versión de Spark: {spark.version}")
print(f"Versión de Python: {os.sys.version}")
print("="*50)

# Prueba simple
from pyspark.sql import Row
test = spark.createDataFrame([Row(prueba=1)])
print("\n✅ Prueba de Spark funcionando:")
test.show()

1. Instalando Java 11...
2. Descargando Spark 3.5.0...
3. Descomprimiendo Spark...
4. Instalando paquetes Python...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
5. Configurando variables de entorno...
6. Inicializando Spark...

✅ CONFIGURACIÓN EXITOSA
Versión de Spark: 3.5.0
Versión de Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

✅ Prueba de Spark funcionando:
+------+
|prueba|
+------+
|     1|
+------+



In [5]:
# ============================================
# ACTIVIDAD COMPLETA - PROCESAMIENTO DE DATOS
# ============================================

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import col, sum as spark_sum, count

print("="*60)
print("INICIANDO ACTIVIDAD 2")
print("="*60)

# 1. DEFINIR ESQUEMA
print("\n📋 1. DEFINICIÓN DEL ESQUEMA")
print("-"*40)
esquema = StructType([
    StructField("Order_ID", StringType(), True),
    StructField("Fecha", StringType(), True),
    StructField("Producto", StringType(), True),
    StructField("Categoria", StringType(), True),
    StructField("Cantidad", IntegerType(), True),
    StructField("Precio", DoubleType(), True),
    StructField("Total", DoubleType(), True)
])
print("✅ Esquema definido correctamente")
print("Campos: Order_ID, Fecha, Producto, Categoria, Cantidad, Precio, Total")

# 2. CREAR DATOS DE EJEMPLO
print("\n📊 2. CREANDO DATOS DE VENTAS")
print("-"*40)
datos = [
    ("ORD-001", "2024-01-15", "Laptop", "Electrónica", 2, 800.0, 1600.0),
    ("ORD-002", "2024-01-15", "Mouse", "Electrónica", 5, 25.0, 125.0),
    ("ORD-003", "2024-01-16", "Silla", "Muebles", 1, 250.0, 250.0),
    ("ORD-004", "2024-01-16", "Monitor", "Electrónica", 1, 300.0, 300.0),
    ("ORD-005", "2024-01-17", "Teclado", "Electrónica", 3, 45.0, 135.0),
    ("ORD-006", "2024-01-17", "Escritorio", "Muebles", 1, 450.0, 450.0),
    ("ORD-007", "2024-01-18", "Celular", "Electrónica", 4, 350.0, 1400.0),
    ("ORD-008", "2024-01-18", "Funda", "Accesorios", 10, 15.0, 150.0),
    ("ORD-009", "2024-01-19", "Cargador", "Electrónica", 6, 20.0, 120.0),
    ("ORD-010", "2024-01-19", "Mochila", "Accesorios", 2, 60.0, 120.0),
]

df = spark.createDataFrame(datos, schema=esquema)
print(f"✅ DataFrame creado con {df.count()} registros")

# 3. MOSTRAR DATOS
print("\n📄 3. VISUALIZACIÓN DE DATOS")
print("-"*40)
print("Primeros 10 registros:")
df.show(10)

# 4. METADATOS (printSchema)
print("\n🔧 4. METADATOS - ESTRUCTURA DE LA TABLA")
print("-"*40)
print("Esquema completo (tipos de datos y nulabilidad):")
df.printSchema()

# 5. ESTADÍSTICAS DESCRIPTIVAS (describe)
print("\n📈 5. ESTADÍSTICAS DESCRIPTIVAS")
print("-"*40)
print("Resumen estadístico de columnas numéricas:")
df.describe().show()

# 6. GROUP BY CON PYSPARK
print("\n📊 6. CONSULTA GROUP BY - PYSPARK")
print("-"*40)
print("Ventas agrupadas por categoría:")
resultado_pyspark = df.groupBy("Categoria").agg(
    count("*").alias("Cantidad_Ventas"),
    spark_sum("Total").alias("Total_Ventas")
).orderBy("Total_Ventas", ascending=False)
resultado_pyspark.show()

# 7. CREAR TABLA TEMPORAL PARA SQL
df.createOrReplaceTempView("ventas")
print("\n✅ Tabla temporal 'ventas' creada para consultas SQL")

# 8. GROUP BY CON SQL
print("\n📊 7. CONSULTA GROUP BY - SQL")
print("-"*40)
print("Misma consulta pero con lenguaje SQL:")
spark.sql("""
    SELECT
        Categoria,
        COUNT(*) as Cantidad_Ventas,
        SUM(Total) as Total_Ventas
    FROM ventas
    GROUP BY Categoria
    ORDER BY Total_Ventas DESC
""").show()

# 9. FILTROS CON PYSPARK
print("\n🔍 8. FILTRO - PYSPARK")
print("-"*40)
print("Ventas mayores a $500:")
df.filter(col("Total") > 500).select("Order_ID", "Producto", "Total").show()

# 10. FILTROS CON SQL
print("\n🔍 9. FILTRO - SQL")
print("-"*40)
print("Misma consulta pero con SQL:")
spark.sql("SELECT Order_ID, Producto, Total FROM ventas WHERE Total > 500").show()

# 11. CONTAR REGISTROS
print("\n🔢 10. TOTAL DE REGISTROS")
print("-"*40)
print(f"La tabla tiene {df.count()} registros en total")



INICIANDO ACTIVIDAD 2

📋 1. DEFINICIÓN DEL ESQUEMA
----------------------------------------
✅ Esquema definido correctamente
Campos: Order_ID, Fecha, Producto, Categoria, Cantidad, Precio, Total

📊 2. CREANDO DATOS DE VENTAS
----------------------------------------
✅ DataFrame creado con 10 registros

📄 3. VISUALIZACIÓN DE DATOS
----------------------------------------
Primeros 10 registros:
+--------+----------+----------+-----------+--------+------+------+
|Order_ID|     Fecha|  Producto|  Categoria|Cantidad|Precio| Total|
+--------+----------+----------+-----------+--------+------+------+
| ORD-001|2024-01-15|    Laptop|Electrónica|       2| 800.0|1600.0|
| ORD-002|2024-01-15|     Mouse|Electrónica|       5|  25.0| 125.0|
| ORD-003|2024-01-16|     Silla|    Muebles|       1| 250.0| 250.0|
| ORD-004|2024-01-16|   Monitor|Electrónica|       1| 300.0| 300.0|
| ORD-005|2024-01-17|   Teclado|Electrónica|       3|  45.0| 135.0|
| ORD-006|2024-01-17|Escritorio|    Muebles|       1| 450.0| 

## 📋 Diccionario de Datos - Ventas E-commerce

| Campo | Tipo de Dato | Descripción | ¿Puede estar vacío? |
|-------|--------------|-------------|---------------------|
| Order_ID | String | Identificador único de cada pedido | No |
| Fecha | String | Día de la venta (formato YYYY-MM-DD) | No |
| Producto | String | Nombre del producto vendido | No |
| Categoria | String | Categoría del producto | No |
| Cantidad | Integer | Número de unidades vendidas | No |
| Precio | Double | Precio por unidad | No |
| Total | Double | Monto total (Cantidad × Precio) | No |

**Justificación de tipos:** El campo `Order_ID` es String porque no se realizan operaciones matemáticas con él. `Total` es Double para permitir decimales en los montos.

## ⚙️ Configuración del Entorno

**Plataforma utilizada:** Google Colab con PySpark

**Versiones:**
- Spark: 3.5.0
- Python: 3.10

**Recursos:**
- Memoria driver: 2GB
- Núcleos: Todos disponibles (local[*])

La configuración permite procesar datasets de tamaño pequeño a mediano de manera eficiente.

## 📊 Análisis de Resultados

### Metadatos (printSchema)
El esquema muestra 7 columnas con sus respectivos tipos de datos. Todas las columnas permiten valores nulos (True), aunque en nuestros datos no hay nulos.

### Estadísticas (describe)
- **Total de registros:** 10 transacciones
- **Precio promedio:** $229.50
- **Total promedio:** $474.50
- **Venta máxima:** $1,600 (Laptop)
- **Venta mínima:** $120 (Cargador y Mochila)

### GROUP BY por Categoría
La categoría Electrónica domina las ventas con $3,680, seguida por Muebles con $700 y Accesorios con $270.

### Filtros
Solo 3 ventas superan los $500: Laptop ($1,600), Celular ($1,400) y Escritorio ($450 - este no supera $500, revisar). Realmente son 2 ventas mayores a $500.

## 🔍 Comparación: SQL vs PySpark

| Aspecto | SQL | PySpark |
|---------|-----|---------|
| **Facilidad de escritura** | ✅ Muy intuitivo | ❌ Más código |
| **Depuración** | ❌ Difícil | ✅ Fácil (print, debug) |
| **Transformaciones complejas** | ❌ Código enredado | ✅ Organizado por pasos |
| **Reutilización de código** | ❌ Limitada | ✅ Funciones y variables |
| **Rendimiento** | ✅ Bueno | ✅✅ Excelente |

### Conclusión
En esta actividad usé **PySpark** para la carga y transformación de datos, y **SQL** para las consultas de agrupación. La combinación de ambos es óptima: PySpark para el procesamiento pesado y SQL para consultas rápidas.

**Recomendación:** Usar PySpark para ETL (extracción, transformación, carga) y SQL para análisis exploratorio de datos.